# Stage 1 — View 검출 (YOLOv11, AABB)

전체 도면에서 **view 영역**(필요 시 표제란 `title_block`)을 축정렬 박스로 검출합니다.

**워크플로**
1. `rasterize_pdf.ipynb` 로 만든 `data/drawings_pages/*.png` 50장을 **CVAT(rectangle)** 로 라벨링
2. CVAT → **"Ultralytics YOLO Detection"** 포맷으로 export → 압축 해제하여 `detection/view/cvat_export/` 에 둠
   (그 안에 `images/`, `labels/` 존재)
3. 아래 split 셀로 `datasets/view/{train,val}` 구성 (45/5, seed 42)
4. 학습 → 검증 → 추론·크롭

- 커널: **kardi_env** (ultralytics 8.4.67)
- 클래스: `detection/view/view.yaml` 참고 (0=view, 1=title_block)

In [ ]:
# ── 환경 확인 ─────────────────────────────────────────────────────
from pathlib import Path
import ultralytics, torch
from ultralytics import YOLO
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available())
HERE = Path.cwd()
assert (HERE / "view.yaml").exists(), f"detection/view 에서 실행하세요 (현재: {HERE})"

In [ ]:
# ── CVAT export → train/val 분리 ─────────────────────────────────
# CVAT 'Ultralytics YOLO Detection' export 압축을 풀어 cvat_export/ 에 두세요.
# (images/, labels/ 폴더 포함). 이미지-라벨 stem 매칭만 사용합니다.
import random, shutil
EXPORT_DIR = Path("cvat_export")
DST        = Path("datasets/view")
VAL_RATIO, SEED = 0.1, 42

src_img = EXPORT_DIR / "images"
src_lbl = EXPORT_DIR / "labels"
assert src_img.exists(), f"{src_img} 없음 — CVAT export 를 여기에 두세요"

stems = sorted(p.stem for p in src_img.glob("*.*")
               if (src_lbl / (p.stem + ".txt")).exists())
random.Random(SEED).shuffle(stems)
n_val = max(1, int(len(stems) * VAL_RATIO))
splits = {"val": stems[:n_val], "train": stems[n_val:]}

for split, names in splits.items():
    for sub in ("images", "labels"):
        (DST / sub / split).mkdir(parents=True, exist_ok=True)
    for st in names:
        img = next(src_img.glob(st + ".*"))
        shutil.copy(img, DST / "images" / split / img.name)
        shutil.copy(src_lbl / (st + ".txt"), DST / "labels" / split / (st + ".txt"))
print({k: len(v) for k, v in splits.items()})

In [ ]:
# ── 학습 (YOLOv11n, 도면 특화 보수적 증강) ────────────────────────
# 도면은 방향이 고정 → 좌우/상하 반전·회전·모자이크 OFF, 밝기만 약하게.
model = YOLO("yolo11n.pt")          # 사전학습 시드 (자동 다운로드)
model.train(
    data="view.yaml", epochs=200, imgsz=1280, batch=4,
    project="runs", name="view", exist_ok=True, seed=42,
    fliplr=0.0, flipud=0.0, degrees=0.0, mosaic=0.0,
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.3, translate=0.05, scale=0.2,
)

In [ ]:
# ── 검증 (mAP) ────────────────────────────────────────────────────
best = YOLO("runs/view/weights/best.pt")
metrics = best.val(data="view.yaml")
print(f"mAP50 = {metrics.box.map50:.3f} | mAP50-95 = {metrics.box.map:.3f}")

In [ ]:
# ── 추론 → view 크롭 저장 (Stage 2 입력) ──────────────────────────
import sys; sys.path.append("..")          # detection/ 의 crop_utils 사용
from crop_utils import save_crops_from_result
best  = YOLO("runs/view/weights/best.pt")
PAGES = Path("../../../data/drawings_pages")
OUT   = Path("../../../data/view_crops")
n = 0
for img in sorted(PAGES.glob("*.png")):
    r = best.predict(img, imgsz=1280, conf=0.25, verbose=False)[0]
    n += len(save_crops_from_result(r, OUT, img.stem, pad=4))
print(f"view 크롭 {n}개 → {OUT}")

In [ ]:
# ── 예측 박스 시각화(육안 검증) ───────────────────────────────────
import matplotlib.pyplot as plt
sample = sorted(Path("../../../data/drawings_pages").glob("*.png"))[0]
r = YOLO("runs/view/weights/best.pt").predict(sample, imgsz=1280, conf=0.25, verbose=False)[0]
plt.figure(figsize=(14, 10)); plt.imshow(r.plot()[..., ::-1]); plt.axis("off")
plt.title(f"view 예측: {sample.name}"); plt.show()